# AVA - Custom Piper Voice (no Drive; local run)

Run top to bottom. Cell 1 RESTARTS the runtime (expected). At **cell 3** an upload button appears - pick the four `videoplayback` files. The finished voice downloads at cell 8.

In [ ]:
# 1) Force Python 3.10 (condacolab) - RESTARTS the runtime
!pip install -q condacolab
import condacolab
condacolab.install_from_url('https://github.com/conda-forge/miniforge/releases/download/24.3.0-0/Miniforge3-24.3.0-0-Linux-x86_64.sh')

In [ ]:
# 2) Install Piper trainer + deps (after the restart)
import condacolab; condacolab.check()
!apt-get -qq install -y espeak-ng ffmpeg >/dev/null
!pip install -q piper-phonemize 2>&1 | tail -2
!git clone -q https://github.com/rhasspy/piper /content/piper
%cd /content/piper/src/python
!pip install -q -e . && pip install -q torchmetrics==0.11.4 'numpy<2' pydub soundfile six
!bash build_monotonic_align.sh
import torch; print('python ok | torch', torch.__version__, '| cuda', torch.cuda.is_available())
!python -c "import piper_train, piper_phonemize, six; print('PIPER OK')"

In [ ]:
# 3) Upload your source audio (click Choose Files, pick the four videoplayback files)
import os
from google.colab import files
os.makedirs('/content/audio', exist_ok=True)
up = files.upload()
for name in up:
    open('/content/audio/'+name, 'wb').write(up[name])
print('uploaded:', sorted(os.listdir('/content/audio')))

In [ ]:
# 4) Build dataset (segment + transcribe -> LJSpeech). faster-whisper installs into the kernel's real Python.
!/usr/bin/python3.real -m pip install -q faster-whisper
import importlib; importlib.invalidate_caches()
import os, re, csv, glob
from pathlib import Path
from pydub import AudioSegment, silence
from pydub.effects import normalize
from faster_whisper import WhisperModel
import torch
SRCDIR = '/content/audio'
OUT = Path('/content/dataset'); WAVS = OUT/'wavs'; WAVS.mkdir(parents=True, exist_ok=True)
SR, MIN_S, MAX_S = 22050, 3.0, 15.0
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
asr = WhisperModel('small.en', device=dev, compute_type='float16' if dev=='cuda' else 'int8')
def segment(a):
    chunks = silence.split_on_silence(a, min_silence_len=500, silence_thresh=-38, keep_silence=200) or [a]
    out, buf = [], AudioSegment.empty()
    for ch in chunks:
        buf += ch
        if len(buf) >= MIN_S*1000:
            while len(buf) > MAX_S*1000:
                out.append(buf[:int(MAX_S*1000)]); buf = buf[int(MAX_S*1000):]
            if len(buf) >= MIN_S*1000: out.append(buf); buf = AudioSegment.empty()
    if len(buf) >= MIN_S*1000: out.append(buf)
    return out
rows, idx, total = [], 0, 0.0
SRC = sorted(glob.glob(SRCDIR+'/*.mp4')+glob.glob(SRCDIR+'/*.m4a')+glob.glob(SRCDIR+'/*.mp3')+glob.glob(SRCDIR+'/*.wav')+glob.glob(SRCDIR+'/*.webm'))
for f in SRC:
    print('processing', os.path.basename(f))
    a = AudioSegment.from_file(f).set_channels(1).set_frame_rate(SR)
    for seg in segment(a):
        seg = normalize(seg).set_channels(1).set_frame_rate(SR).set_sample_width(2)
        wid = 'ava_%05d' % idx; wp = WAVS/(wid+'.wav'); seg.export(wp, format='wav')
        segs,_ = asr.transcribe(str(wp), language='en', beam_size=5)
        txt = re.sub(r'\s+',' ',' '.join(s.text for s in segs)).strip()
        if len(txt) < 2 or not re.search(r'[A-Za-z]', txt):
            wp.unlink(missing_ok=True); continue
        rows.append((wid, txt)); total += len(seg)/1000.0; idx += 1
        if idx % 50 == 0: print(' ', idx, 'clips')
with open(OUT/'metadata.csv','w',encoding='utf-8',newline='') as fh:
    csv.writer(fh, delimiter='|').writerows(rows)
print('DONE:', len(rows), 'clips, ~%.1f min' % (total/60))
!head -n 5 /content/dataset/metadata.csv

In [ ]:
# 5) Piper preprocess
!cd /content/piper/src/python && python -m piper_train.preprocess --language en-us --input-dir /content/dataset --output-dir /content/train --dataset-format ljspeech --single-speaker --sample-rate 22050

In [ ]:
# 6) Download base checkpoint (en_US-lessac-medium)
!wget -q --show-progress -O /content/base.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch=2164-step=1355540.ckpt"
!ls -lh /content/base.ckpt

In [ ]:
# 7) Fine-tune (checkpoints save to /content/train every 5 epochs)
!cd /content/piper/src/python && python -m piper_train --dataset-dir /content/train --accelerator gpu --devices 1 --batch-size 12 --validation-split 0.05 --num-test-examples 2 --max_epochs 3000 --resume_from_checkpoint /content/base.ckpt --checkpoint-epochs 5 --precision 32

In [ ]:
# 8) Export newest checkpoint -> ONNX + download (run any time; re-runnable)
import glob, os
cks = sorted(glob.glob('/content/train/lightning_logs/**/checkpoints/*.ckpt', recursive=True), key=os.path.getmtime)
assert cks, 'No checkpoint yet - let training run longer first.'
ck = cks[-1]; print('exporting', ck)
!cd /content/piper/src/python && python -m piper_train.export_onnx "{ck}" /content/ava_southern.onnx
!cp /content/train/config.json /content/ava_southern.onnx.json
from google.colab import files
files.download('/content/ava_southern.onnx')
files.download('/content/ava_southern.onnx.json')